# SBA 7(a) — Feature Integrity, Drift, and Simulator Exports

This notebook examines feature distributions, interest-rate drift, historical calibration, and interest-rate sensitivity using the **frozen, uncalibrated 13-feature Logistic Regression model**. It also exports the two simulator CSV files. Run from the `notebooks/` working directory with Python 3.12.8 and scikit-learn 1.5.1, after producing `data/exports/sba_model_export.csv` and `models/logistic_regression_13_features.joblib`. Output tables and charts are historical records and were not recomputed during this review. Interest-rate replacement is a **model sensitivity diagnostic, not a causal experiment**. The loan data must remain local if excluded by `.gitignore`.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

DATA_PATH = Path("../data/exports/sba_model_export.csv")

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Missing model export: {DATA_PATH}. Generate it from PostgreSQL first, "
        "then run this notebook from notebooks/."
    )

df = pd.read_csv(DATA_PATH)

numeric_features = [
    "gross_approval",
    "guarantee_pct",
    "initial_interest_rate_model",
    "jobs_supported"
]

flag_features = [
    "invalid_interest_rate_flag"
]

target_column = "chargeoff_within_5yr"

print("Dataset shape:", df.shape)
print("Numeric predictors:", numeric_features)
print("Flag predictors:", flag_features)

assert df.shape == (478953, 25)
assert all(
    column in df.columns
    for column in numeric_features + flag_features
)

print("\nFeature audit setup validated!")

Dataset shape: (478953, 25)
Numeric predictors: ['gross_approval', 'guarantee_pct', 'initial_interest_rate_model', 'jobs_supported']
Flag predictors: ['invalid_interest_rate_flag']

Feature audit setup validated!


In [2]:
# Step 86.2.1 — Missing-value audit

audit_features = numeric_features + flag_features

missing_summary = pd.DataFrame({
    "missing_count": df[audit_features].isna().sum(),
    "missing_pct": (
        df[audit_features].isna().mean() * 100
    )
})

display(missing_summary.round(3))

,missing_count,missing_pct
gross_approval,0,0.000
guarantee_pct,0,0.000
initial_interest_rate_model,7,0.001
jobs_supported,0,0.000
invalid_interest_rate_flag,0,0.000


In [3]:
# Step 86.2.2 — Descriptive statistics

numeric_summary = (
    df[numeric_features]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
    .T
)

display(numeric_summary.round(3))

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
gross_approval,478953.0,376761.109,671179.990,1000.00,5000.00,10000.0,40000.0,125000.0,360000.0,1693000.00,3545000.0,5000000.0
guarantee_pct,478953.0,64.989,15.088,10.25,50.00,50.0,50.0,75.0,75.0,90.00,90.0,100.0
initial_interest_rate_model,478946.0,6.545,1.558,0.66,3.75,4.5,5.5,6.0,7.5,9.75,11.0,13.5
jobs_supported,478953.0,10.717,20.497,0.00,0.00,0.0,2.0,4.0,11.0,41.00,100.0,2150.0


In [4]:
# Step 86.3.1 — Extreme-value outcome audit

outlier_results = []

for feature in numeric_features:

    q01 = df[feature].quantile(0.01)
    q99 = df[feature].quantile(0.99)

    groups = {
        "Lower tail": df[feature] <= q01,
        "Middle": (
            (df[feature] > q01)
            & (df[feature] < q99)
        ),
        "Upper tail": df[feature] >= q99,
        "Missing": df[feature].isna()
    }

    for group_name, mask in groups.items():

        subset = df.loc[mask]

        if subset.empty:
            continue

        outlier_results.append({
            "feature": feature,
            "group": group_name,
            "loan_count": len(subset),
            "chargeoffs": int(
                subset[target_column].sum()
            ),
            "chargeoff_rate_pct": (
                subset[target_column].mean() * 100
            ),
            "min_value": subset[feature].min(),
            "max_value": subset[feature].max()
        })

outlier_summary = pd.DataFrame(outlier_results)

display(outlier_summary.round(3))

# Verify all records are assigned to exactly one group
for feature in numeric_features:
    feature_total = outlier_summary.loc[
        outlier_summary["feature"] == feature,
        "loan_count"
    ].sum()

    assert feature_total == len(df)

print("\nExtreme-value audit validated!")

,feature,group,loan_count,chargeoffs,chargeoff_rate_pct,min_value,max_value
0,gross_approval,Lower tail,7642,588,7.694,1000.000,5000.00
1,gross_approval,Middle,466513,18213,3.904,5100.000,3544000.00
2,gross_approval,Upper tail,4798,25,0.521,3545000.000,5000000.00
3,guarantee_pct,Lower tail,231395,9689,4.187,10.250,50.00
4,guarantee_pct,Middle,219315,8013,3.654,50.130,89.99
5,guarantee_pct,Upper tail,28243,1124,3.980,90.000,100.00
6,initial_interest_rate_model,Lower tail,5078,94,1.851,0.660,3.75
7,initial_interest_rate_model,Middle,468246,18205,3.888,3.754,10.99
8,initial_interest_rate_model,Upper tail,5622,527,9.374,11.000,13.50
9,initial_interest_rate_model,Missing,7,0,0.000,NaN,NaN



Extreme-value audit validated!


In [5]:
# Step 86.4.1 — Audit exact high interest-rate values

high_rate_df = df.loc[
    df["initial_interest_rate_model"] >= 11
].copy()

high_rate_summary = (
    high_rate_df
    .groupby("initial_interest_rate_model")
    .agg(
        loan_count=(target_column, "size"),
        chargeoffs=(target_column, "sum")
    )
    .reset_index()
)

high_rate_summary["chargeoff_rate_pct"] = (
    high_rate_summary["chargeoffs"]
    / high_rate_summary["loan_count"]
    * 100
)

high_rate_summary = high_rate_summary.sort_values(
    "initial_interest_rate_model"
)

display(high_rate_summary.round(3))

print("Total high-interest loans:", len(high_rate_df))
print(
    "Total high-interest charge-offs:",
    int(high_rate_df[target_column].sum())
)

assert len(high_rate_df) == 5622
assert high_rate_df[target_column].sum() == 527

print("\nHigh-interest-rate audit validated!")

,initial_interest_rate_model,loan_count,chargeoffs,chargeoff_rate_pct
0,11.000,954,88,9.224
1,11.020,1,0,0.000
2,11.040,138,20,14.493
3,11.070,1,0,0.000
4,11.100,2,0,0.000
5,11.120,1,0,0.000
6,11.130,114,19,16.667
7,11.150,2,1,50.000
8,11.190,1,0,0.000
9,11.200,3,0,0.000


Total high-interest loans: 5622
Total high-interest charge-offs: 527

High-interest-rate audit validated!


In [6]:
# Step 86.5.1 — High-interest risk across fiscal years

df["high_interest_flag"] = (
    df["initial_interest_rate_model"] >= 11
)

interest_fy_summary = (
    df.groupby(["approval_fy", "high_interest_flag"])
    .agg(
        loan_count=(target_column, "size"),
        chargeoffs=(target_column, "sum")
    )
    .reset_index()
)

interest_fy_summary["chargeoff_rate_pct"] = (
    interest_fy_summary["chargeoffs"]
    / interest_fy_summary["loan_count"]
    * 100
)

display(interest_fy_summary.round(3))

,approval_fy,high_interest_flag,loan_count,chargeoffs,chargeoff_rate_pct
0,2010,False,39912,1955,4.898
1,2011,False,45628,1667,3.653
2,2012,False,38887,1276,3.281
3,2012,True,2,0,0.000
4,2013,False,40416,1242,3.073
5,2014,False,45961,1617,3.518
6,2015,False,55420,2137,3.856
7,2016,False,56785,2198,3.871
8,2017,False,56077,2419,4.314
9,2017,True,1,0,0.000


In [7]:
# Compare high-interest and other loans by fiscal year

interest_fy_pivot = interest_fy_summary.pivot(
    index="approval_fy",
    columns="high_interest_flag",
    values=["loan_count", "chargeoff_rate_pct"]
)

interest_fy_pivot.columns = [
    f"{metric}_{'High_Interest' if flag else 'Other'}"
    for metric, flag in interest_fy_pivot.columns
]

display(interest_fy_pivot.round(3))

,loan_count_Other,loan_count_High_Interest,chargeoff_rate_pct_Other,chargeoff_rate_pct_High_Interest
approval_fy,,,,
2010,39912.0,NaN,4.898,NaN
2011,45628.0,NaN,3.653,NaN
2012,38887.0,2.0,3.281,0.000
2013,40416.0,NaN,3.073,NaN
2014,45961.0,NaN,3.518,NaN
2015,55420.0,NaN,3.856,NaN
2016,56785.0,NaN,3.871,NaN
2017,56077.0,1.0,4.314,0.000
2018,52211.0,1983.0,4.436,10.741


In [8]:
# Step 86.6.1 — Interest-rate distribution by approval FY

interest_distribution = (
    df.groupby("approval_fy")[
        "initial_interest_rate_model"
    ]
    .agg(
        loan_count="count",
        mean="mean",
        median="median",
        min="min",
        max="max",
        p95=lambda x: x.quantile(0.95),
        p99=lambda x: x.quantile(0.99)
    )
    .reset_index()
)

display(interest_distribution.round(3))

,approval_fy,loan_count,mean,median,min,max,p95,p99
0,2010,39912,6.451,6.00,0.66,10.00,8.75,9.75
1,2011,45628,6.299,6.00,1.57,10.70,8.75,9.75
2,2012,38889,6.232,6.00,1.00,11.52,9.15,9.75
3,2013,40416,6.114,5.95,1.00,10.94,9.50,9.75
4,2014,45961,6.037,5.85,1.00,9.75,9.25,9.75
5,2015,55420,6.119,5.90,1.00,9.75,9.75,9.75
6,2016,56783,6.271,6.00,1.01,10.02,9.75,10.00
7,2017,56076,6.624,6.25,1.25,11.04,10.00,10.74
8,2018,54192,7.291,7.00,0.75,12.50,10.75,11.50
9,2019,45669,7.909,7.75,1.30,13.50,11.50,12.00


In [9]:
# Step 86.7.1 — Training-range audit

rate_col = "initial_interest_rate_model"

# Frozen model was trained on FY2010–FY2015
train_rates = df.loc[
    df["approval_fy"].between(2010, 2015),
    rate_col
].dropna()

train_min = train_rates.min()
train_max = train_rates.max()

print(f"Training minimum rate: {train_min:.3f}%")
print(f"Training maximum rate: {train_max:.3f}%")

Training minimum rate: 0.660%
Training maximum rate: 11.520%


In [10]:
# Step 86.7.2 — Out-of-training-range rates

range_results = []

for fy in [2016, 2017, 2018, 2019]:

    cohort = df.loc[
        df["approval_fy"] == fy
    ].copy()

    above_train = cohort[rate_col] > train_max
    below_train = cohort[rate_col] < train_min

    range_results.append({
        "approval_fy": fy,
        "total_loans": len(cohort),
        "above_train_max": int(above_train.sum()),
        "above_train_max_pct": above_train.mean() * 100,
        "below_train_min": int(below_train.sum()),
        "below_train_min_pct": below_train.mean() * 100
    })

range_summary = pd.DataFrame(range_results)

display(range_summary.round(3))

,approval_fy,total_loans,above_train_max,above_train_max_pct,below_train_min,below_train_min_pct
0,2016,56785,0,0.000,0,0.0
1,2017,56078,0,0.000,0,0.0
2,2018,54194,4,0.007,0,0.0
3,2019,45670,2228,4.878,0,0.0


In [11]:
# Step 86.8.1 — Interest-rate flag audit

flag_summary = (
    df.groupby(
        "invalid_interest_rate_flag",
        dropna=False
    )
    .agg(
        loan_count=(target_column, "size"),
        chargeoffs=(target_column, "sum"),
        missing_interest_rates=(
            "initial_interest_rate_model",
            lambda x: x.isna().sum()
        )
    )
    .reset_index()
)

flag_summary["chargeoff_rate_pct"] = (
    flag_summary["chargeoffs"]
    / flag_summary["loan_count"]
    * 100
)

display(flag_summary.round(3))

,invalid_interest_rate_flag,loan_count,chargeoffs,missing_interest_rates,chargeoff_rate_pct
0,0,478946,18826,0,3.931
1,1,7,0,7,0.000


In [12]:
# Step 86.8.2 — Missing-rate consistency

missing_rate = df[
    "initial_interest_rate_model"
].isna()

invalid_flag = (
    df["invalid_interest_rate_flag"] == 1
)

print("Missing modeled interest rates:", missing_rate.sum())

print(
    "Missing rates with invalid flag:",
    (missing_rate & invalid_flag).sum()
)

print(
    "Missing rates without invalid flag:",
    (missing_rate & ~invalid_flag).sum()
)

print(
    "Invalid flags with nonmissing modeled rates:",
    (~missing_rate & invalid_flag).sum()
)

Missing modeled interest rates: 7
Missing rates with invalid flag: 7
Missing rates without invalid flag: 0
Invalid flags with nonmissing modeled rates: 0


In [13]:
# Step 86.9.1 — Loan amount and charge-off analysis

amount_bins = [
    0,
    25000,
    50000,
    100000,
    250000,
    500000,
    1000000,
    2000000,
    np.inf
]

amount_labels = [
    "$0–25K",
    "$25K–50K",
    "$50K–100K",
    "$100K–250K",
    "$250K–500K",
    "$500K–1M",
    "$1M–2M",
    "$2M+"
]

df["loan_amount_band"] = pd.cut(
    df["gross_approval"],
    bins=amount_bins,
    labels=amount_labels,
    include_lowest=True,
    right=False
)

amount_summary = (
    df.groupby(
        "loan_amount_band",
        observed=True
    )
    .agg(
        loan_count=(target_column, "size"),
        chargeoffs=(target_column, "sum")
    )
    .reset_index()
)

amount_summary["chargeoff_rate_pct"] = (
    amount_summary["chargeoffs"]
    / amount_summary["loan_count"]
    * 100
)

display(amount_summary.round(3))

assert amount_summary["loan_count"].sum() == len(df)
assert amount_summary["chargeoffs"].sum() == 18826

print("\nLoan amount band audit validated!")

,loan_amount_band,loan_count,chargeoffs,chargeoff_rate_pct
0,$0–25K,63366,4000,6.313
1,$25K–50K,65137,3519,5.402
2,$50K–100K,72933,3155,4.326
3,$100K–250K,108366,4931,4.550
4,$250K–500K,72566,2181,3.006
5,$500K–1M,46620,632,1.356
6,$1M–2M,31728,297,0.936
7,$2M+,18237,111,0.609



Loan amount band audit validated!


In [14]:
# Step 86.10.1 — Guarantee percentage audit

guarantee_summary = (
    df.groupby("guarantee_pct")
    .agg(
        loan_count=(target_column, "size"),
        chargeoffs=(target_column, "sum")
    )
    .reset_index()
)

guarantee_summary["chargeoff_rate_pct"] = (
    guarantee_summary["chargeoffs"]
    / guarantee_summary["loan_count"]
    * 100
)

# Display the 20 most common guarantee percentages
common_guarantees = (
    guarantee_summary
    .sort_values(
        "loan_count",
        ascending=False
    )
    .head(20)
)

display(common_guarantees.round(3))

print(
    "Number of distinct guarantee percentages:",
    df["guarantee_pct"].nunique()
)

# Validate totals
assert guarantee_summary["loan_count"].sum() == len(df)
assert guarantee_summary["chargeoffs"].sum() == 18826

print("\nGuarantee percentage audit validated!")

,guarantee_pct,loan_count,chargeoffs,chargeoff_rate_pct
114,50.00,231167,9688,4.191
442,75.00,161081,3580,2.222
571,85.00,57213,4424,7.733
636,90.00,28242,1124,3.980
503,80.00,40,0,0.000
194,60.00,35,0,0.000
323,70.00,28,1,3.571
245,65.00,24,0,0.000
389,73.00,22,0,0.000
20,25.00,19,0,0.000


Number of distinct guarantee percentages: 638

Guarantee percentage audit validated!


In [15]:
# Step 86.11.1 — Profile common guarantee percentages

common_values = [50.0, 75.0, 85.0, 90.0]

guarantee_profile = (
    df.loc[df["guarantee_pct"].isin(common_values)]
    .groupby("guarantee_pct")
    .agg(
        loan_count=(target_column, "size"),
        chargeoffs=(target_column, "sum"),
        median_loan_amount=("gross_approval", "median"),
        mean_loan_amount=("gross_approval", "mean"),
        median_interest_rate=(
            "initial_interest_rate_model", "median"
        ),
        median_jobs=("jobs_supported", "median")
    )
    .reset_index()
)

guarantee_profile["chargeoff_rate_pct"] = (
    guarantee_profile["chargeoffs"]
    / guarantee_profile["loan_count"]
    * 100
)

display(guarantee_profile.round(3))

,guarantee_pct,loan_count,chargeoffs,median_loan_amount,mean_loan_amount,median_interest_rate,median_jobs,chargeoff_rate_pct
0,50.0,231167,9688,50000.0,76516.501,6.75,3.0,4.191
1,75.0,161081,3580,510000.0,857883.171,6.00,8.0,2.222
2,85.0,57213,4424,100000.0,96365.584,6.25,4.0,7.733
3,90.0,28242,1124,325750.0,595070.342,6.00,7.0,3.980


In [16]:
# Step 86.11.2 — Guarantee risk by approval year

guarantee_fy = (
    df.loc[df["guarantee_pct"].isin(common_values)]
    .groupby(["approval_fy", "guarantee_pct"])
    .agg(
        loan_count=(target_column, "size"),
        chargeoffs=(target_column, "sum")
    )
    .reset_index()
)

guarantee_fy["chargeoff_rate_pct"] = (
    guarantee_fy["chargeoffs"]
    / guarantee_fy["loan_count"]
    * 100
)

display(guarantee_fy.round(3))

assert guarantee_fy["loan_count"].sum() == (
    df["guarantee_pct"].isin(common_values).sum()
)

print("\nGuarantee cohort audit validated!")

,approval_fy,guarantee_pct,loan_count,chargeoffs,chargeoff_rate_pct
0,2010,50.0,16947,649,3.830
1,2010,75.0,3518,93,2.644
2,2010,85.0,3876,465,11.997
3,2010,90.0,15110,745,4.931
4,2011,50.0,22629,812,3.588
5,2011,75.0,8535,205,2.402
6,2011,85.0,4202,322,7.663
7,2011,90.0,10217,327,3.201
8,2012,50.0,20108,736,3.660
9,2012,75.0,13992,292,2.087



Guarantee cohort audit validated!


In [17]:
# Step 86.12.1 — Jobs-supported outcome audit

job_bins = [
    -1,
    0,
    1,
    5,
    10,
    25,
    50,
    100,
    250,
    500,
    np.inf
]

job_labels = [
    "0 jobs",
    "1 job",
    "2–5 jobs",
    "6–10 jobs",
    "11–25 jobs",
    "26–50 jobs",
    "51–100 jobs",
    "101–250 jobs",
    "251–500 jobs",
    "501+ jobs"
]

df["jobs_band"] = pd.cut(
    df["jobs_supported"],
    bins=job_bins,
    labels=job_labels,
    include_lowest=False,
    right=True
)

jobs_summary = (
    df.groupby("jobs_band", observed=True)
    .agg(
        loan_count=(target_column, "size"),
        chargeoffs=(target_column, "sum"),
        median_loan_amount=("gross_approval", "median")
    )
    .reset_index()
)

jobs_summary["chargeoff_rate_pct"] = (
    jobs_summary["chargeoffs"]
    / jobs_summary["loan_count"]
    * 100
)

display(jobs_summary.round(3))

# Validate the audit
assert jobs_summary["loan_count"].sum() == len(df)
assert jobs_summary["chargeoffs"].sum() == 18826

print("\nJobs-supported audit validated!")

,jobs_band,loan_count,chargeoffs,median_loan_amount,chargeoff_rate_pct
0,0 jobs,75321,2821,50000.0,3.745
1,1 job,40127,1919,50000.0,4.782
2,2–5 jobs,151791,7048,100000.0,4.643
3,6–10 jobs,85432,3311,150000.0,3.876
4,11–25 jobs,78953,2628,265000.0,3.329
5,26–50 jobs,31534,808,379000.0,2.562
6,51–100 jobs,11723,236,500000.0,2.013
7,101–250 jobs,3926,53,1000000.0,1.350
8,251–500 jobs,140,2,600000.0,1.429
9,501+ jobs,6,0,800000.0,0.000



Jobs-supported audit validated!


In [4]:
from pathlib import Path
import joblib

MODEL_PATH = Path(
    "../models/logistic_regression_13_features.joblib"
)

import sklearn
if sklearn.__version__ != "1.5.1":
    raise RuntimeError(
        f"Saved model requires scikit-learn 1.5.1; found {sklearn.__version__}."
    )
if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"Missing frozen model: {MODEL_PATH}")

audit_model = joblib.load(MODEL_PATH)

# Access the fitted Logistic Regression classifier
audit_classifier = audit_model.named_steps["classifier"]

print("Model type:", type(audit_classifier).__name__)
print("Class weight:", audit_classifier.class_weight)
print("Regularization C:", audit_classifier.C)
print("Fit intercept:", audit_classifier.fit_intercept)

# Verify the expected selected configuration
assert audit_classifier.C == 1.0
assert audit_classifier.fit_intercept is True

print("\nModel configuration audit completed!")

Model type: LogisticRegression
Class weight: None
Regularization C: 1.0
Fit intercept: True

Model configuration audit completed!


In [1]:
import sklearn
import sys

print("Current scikit-learn version:", sklearn.__version__)
print("Model training version: 1.5.1")
print("Python version:", sys.version.split()[0])

if sklearn.__version__ == "1.5.1":
    print("\nEnvironment version matches!")
else:
    print("\nVersion mismatch detected.")

Current scikit-learn version: 1.5.1
Model training version: 1.5.1
Python version: 3.12.8

Environment version matches!


In [8]:
import pandas as pd
from pathlib import Path

# Load the modeling dataset
DATA_PATH = Path("../data/exports/sba_model_export.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

# Verify dataset integrity
assert df.shape == (478953, 25)

print("Dataset loaded successfully!")

Dataset shape: (478953, 25)
Dataset loaded successfully!


In [9]:
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss
)

# Retrieve the original 13 predictors from the fitted pipeline
feature_columns = list(audit_model.feature_names_in_)

# Select FY2019 loans
test_audit = df.loc[
    df["approval_fy"] == 2019
].copy()

X_audit = test_audit[feature_columns]
y_audit = test_audit["chargeoff_within_5yr"]

# Predict using the saved model (no retraining)
audit_probs = audit_model.predict_proba(X_audit)[:, 1]

# Calculate the original evaluation metrics
audit_ap = average_precision_score(y_audit, audit_probs)
audit_roc = roc_auc_score(y_audit, audit_probs)
audit_brier = brier_score_loss(y_audit, audit_probs)

print("FY2019 Reproducibility Check")
print(f"Loans: {len(test_audit):,}")
print(f"Average predicted risk: {audit_probs.mean():.4%}")
print(f"Average Precision: {audit_ap:.4f}")
print(f"ROC-AUC: {audit_roc:.4f}")
print(f"Brier Score: {audit_brier:.6f}")

# Validate against previously recorded results
assert len(test_audit) == 45670
assert abs(audit_probs.mean() - 0.061107) < 0.00001
assert round(audit_ap, 4) == 0.0774
assert round(audit_roc, 4) == 0.7020
assert abs(audit_brier - 0.037618) < 0.000001

print("\nModel reproducibility check passed!")

FY2019 Reproducibility Check
Loans: 45,670
Average predicted risk: 6.1107%
Average Precision: 0.0774
ROC-AUC: 0.7020
Brier Score: 0.037618

Model reproducibility check passed!


In [11]:
# Step 87.4.1 — Generate frozen-model scores across all fiscal years
import numpy as np
# Retrieve the model's original feature names and order
feature_columns = list(audit_model.feature_names_in_)

# Prepare the full modeling population
X_all = df[feature_columns].copy()

# Generate predictions without retraining
all_probabilities = audit_model.predict_proba(X_all)[:, 1]

# Create a separate analysis DataFrame
calibration_audit = pd.DataFrame({
    "approval_fy": df["approval_fy"].to_numpy(),
    "actual": df["chargeoff_within_5yr"].to_numpy(),
    "predicted": all_probabilities
})

# Validate probabilities
assert len(calibration_audit) == 478953
assert np.isfinite(all_probabilities).all()
assert ((all_probabilities >= 0) & (all_probabilities <= 1)).all()

print("Loans scored:", len(calibration_audit))
print("All-cohort prediction validation passed!")

Loans scored: 478953
All-cohort prediction validation passed!


In [13]:
# Step 87.4.2 — Annual calibration summary

annual_calibration = (
    calibration_audit
    .groupby("approval_fy")
    .agg(
        loan_count=("actual", "size"),
        chargeoffs=("actual", "sum"),
        observed_rate=("actual", "mean"),
        avg_predicted_risk=("predicted", "mean")
    )
    .reset_index()
)

annual_calibration["overprediction_gap_pp"] = (
    annual_calibration["avg_predicted_risk"]
    - annual_calibration["observed_rate"]
) * 100

annual_calibration["observed_rate_pct"] = (
    annual_calibration["observed_rate"] * 100
)

annual_calibration["avg_predicted_risk_pct"] = (
    annual_calibration["avg_predicted_risk"] * 100
)

display(
    annual_calibration[
        [
            "approval_fy",
            "loan_count",
            "chargeoffs",
            "observed_rate_pct",
            "avg_predicted_risk_pct",
            "overprediction_gap_pp"
        ]
    ].round(3)
)

print("Annual calibration table created successfully!")

,approval_fy,loan_count,chargeoffs,observed_rate_pct,avg_predicted_risk_pct,overprediction_gap_pp
0,2010,39912,1955,4.898,4.888,-0.011
1,2011,45628,1667,3.653,3.837,0.184
2,2012,38889,1276,3.281,3.410,0.129
3,2013,40416,1242,3.073,3.423,0.350
4,2014,45961,1617,3.518,3.356,-0.162
5,2015,55420,2137,3.856,3.490,-0.366
6,2016,56785,2198,3.871,3.648,-0.222
7,2017,56078,2419,4.314,3.741,-0.573
8,2018,54194,2529,4.667,5.153,0.486
9,2019,45670,1786,3.911,6.111,2.200


Annual calibration table created successfully!


In [14]:
# Step 87.4.3 — Validate annual calibration results

fy2019 = annual_calibration.loc[
    annual_calibration["approval_fy"] == 2019
].iloc[0]

assert len(annual_calibration) == 10
assert annual_calibration["loan_count"].sum() == 478953
assert annual_calibration["chargeoffs"].sum() == 18826

assert abs(
    fy2019["avg_predicted_risk"] - 0.061107
) < 0.00001

print("Annual calibration audit validated!")


Annual calibration audit validated!


In [15]:
# Step 87.5.1 — Reference interest rate

train_median_rate = df.loc[
    df["approval_fy"].between(2010, 2015),
    "initial_interest_rate_model"
].median()

print(
    "FY2010–2015 median interest rate:",
    f"{train_median_rate:.3f}%"
)

FY2010–2015 median interest rate: 6.000%


In [16]:
# Step 87.5.2 — Interest-rate sensitivity analysis

fy2019 = df.loc[
    df["approval_fy"] == 2019
].copy()

X_original = fy2019[feature_columns].copy()

# Create a diagnostic copy
X_reference_rate = X_original.copy()

# Replace only the interest-rate predictor
X_reference_rate["initial_interest_rate_model"] = (
    train_median_rate
)

# Frozen model predictions
original_probs = audit_model.predict_proba(
    X_original
)[:, 1]

reference_rate_probs = audit_model.predict_proba(
    X_reference_rate
)[:, 1]

print("FY2019 Interest-Rate Sensitivity")

print(
    "Original average prediction:",
    f"{original_probs.mean():.4%}"
)

print(
    "Reference-rate average prediction:",
    f"{reference_rate_probs.mean():.4%}"
)

print(
    "Prediction change:",
    f"{(reference_rate_probs.mean() - original_probs.mean()) * 100:.3f}",
    "percentage points"
)

print(
    "Observed FY2019 incidence:",
    f"{fy2019['chargeoff_within_5yr'].mean():.4%}"
)

assert abs(original_probs.mean() - 0.061107) < 0.00001

print("\nInterest-rate sensitivity analysis validated!")

FY2019 Interest-Rate Sensitivity
Original average prediction: 6.1107%
Reference-rate average prediction: 3.6121%
Prediction change: -2.499 percentage points
Observed FY2019 incidence: 3.9107%

Interest-rate sensitivity analysis validated!


In [17]:
# Step 87.6.1 — Temporal interest-rate sensitivity

sensitivity_results = []

for fy in [2016, 2017, 2018, 2019]:

    cohort = df.loc[
        df["approval_fy"] == fy
    ].copy()

    X_original = cohort[feature_columns].copy()

    X_reference = X_original.copy()

    # Hold interest rates at the training median
    X_reference["initial_interest_rate_model"] = (
        train_median_rate
    )

    # Predictions from the frozen model
    original_probs = audit_model.predict_proba(
        X_original
    )[:, 1]

    reference_probs = audit_model.predict_proba(
        X_reference
    )[:, 1]

    sensitivity_results.append({
        "approval_fy": fy,
        "loan_count": len(cohort),
        "observed_rate_pct": (
            cohort["chargeoff_within_5yr"].mean() * 100
        ),
        "original_prediction_pct": (
            original_probs.mean() * 100
        ),
        "reference_prediction_pct": (
            reference_probs.mean() * 100
        ),
        "rate_sensitivity_pp": (
            original_probs.mean()
            - reference_probs.mean()
        ) * 100
    })

sensitivity_summary = pd.DataFrame(sensitivity_results)

display(sensitivity_summary.round(3))

assert len(sensitivity_summary) == 4
assert sensitivity_summary["loan_count"].sum() == (
    56785 + 56078 + 54194 + 45670
)

print("\nTemporal interest-rate sensitivity validated!")

,approval_fy,loan_count,observed_rate_pct,original_prediction_pct,reference_prediction_pct,rate_sensitivity_pp
0,2016,56785,3.871,3.648,3.087,0.561
1,2017,56078,4.314,3.741,2.912,0.829
2,2018,54194,4.667,5.153,3.467,1.686
3,2019,45670,3.911,6.111,3.612,2.499



Temporal interest-rate sensitivity validated!


In [18]:
# Step 87.7 — Export post-modeling calibration diagnostics

RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

annual_path = RESULTS_DIR / "annual_calibration_drift.csv"
sensitivity_path = RESULTS_DIR / "interest_rate_sensitivity.csv"

annual_calibration.to_csv(
    annual_path,
    index=False
)

sensitivity_summary.to_csv(
    sensitivity_path,
    index=False
)

assert annual_path.exists()
assert sensitivity_path.exists()

print("Annual calibration audit saved:", annual_path.exists())
print("Interest-rate sensitivity saved:", sensitivity_path.exists())

print("\nStep 87.7 exports validated!")

Annual calibration audit saved: True
Interest-rate sensitivity saved: True

Step 87.7 exports validated!


In [19]:
# Step 90.1 — Export FY2019 loan-level simulator data

from pathlib import Path
import pandas as pd
import numpy as np

simulator_df = df.loc[
    df["approval_fy"] == 2019
].copy()

# Use the exact frozen feature set
X_simulator = simulator_df[
    list(audit_model.feature_names_in_)
].copy()

# Generate original, uncalibrated model probabilities
simulator_df["predicted_risk_score"] = (
    audit_model.predict_proba(X_simulator)[:, 1]
)

# Select fields relevant to the Excel simulator
simulator_columns = [
    "loan_row_id",
    "approval_fy",
    "gross_approval",
    "guarantee_pct",
    "processing_method",
    "naics_sector",
    "project_state",
    "chargeoff_within_5yr",
    "predicted_risk_score"
]

simulator_export = simulator_df[
    simulator_columns
].copy()

# Sort loans by descending risk score
simulator_export = simulator_export.sort_values(
    ["predicted_risk_score", "loan_row_id"],
    ascending=[False, True],
    kind="stable"
).reset_index(drop=True)

# Validate the export
assert len(simulator_export) == 45670
assert simulator_export["loan_row_id"].is_unique
assert simulator_export["chargeoff_within_5yr"].sum() == 1786
assert abs(
    simulator_export["predicted_risk_score"].mean()
    - 0.061107
) < 0.00001

# Confirm historical top-10% capture
top_10_count = int(np.ceil(len(simulator_export) * 0.10))

captured_top_10 = int(
    simulator_export.head(top_10_count)[
        "chargeoff_within_5yr"
    ].sum()
)

assert captured_top_10 == 398

# Save the simulator input
export_path = Path(
    "../data/exports/fy2019_risk_simulator_input.csv"
)

export_path.parent.mkdir(parents=True, exist_ok=True)
simulator_export.to_csv(export_path, index=False)

print("Export saved:", export_path.exists())
print("Loans:", len(simulator_export))
print("Historical charge-offs:", int(simulator_export["chargeoff_within_5yr"].sum()))
print("Top-10% charge-offs captured:", captured_top_10)
print("Columns:", len(simulator_export.columns))

print("\nExcel simulator dataset validated!")

Export saved: True
Loans: 45670
Historical charge-offs: 1786
Top-10% charge-offs captured: 398
Columns: 9

Excel simulator dataset validated!


In [20]:
# Step 92 — Expanded simulator data export

from pathlib import Path
import numpy as np
import pandas as pd

# Select the validation and final-test cohorts
sim_df = df.loc[
    df["approval_fy"].isin([2018, 2019])
].copy()

# Use the frozen model's exact feature set
X_sim = sim_df[
    list(audit_model.feature_names_in_)
].copy()

sim_df["predicted_risk_score"] = (
    audit_model.predict_proba(X_sim)[:, 1]
)

# Preserve cohort and loan-level audit fields
simulator_columns = [
    "loan_row_id",
    "approval_fy",
    "gross_approval",
    "guarantee_pct",
    "processing_method",
    "naics_sector",
    "project_state",
    "lender_key",
    "gross_chargeoff_amount",
    "chargeoff_within_5yr",
    "predicted_risk_score"
]

expanded_export = sim_df[simulator_columns].copy()

# Convert reported charge-off amounts to numeric
expanded_export["gross_chargeoff_amount"] = (
    pd.to_numeric(
        expanded_export["gross_chargeoff_amount"],
        errors="coerce"
    )
)

# Sort within each cohort by risk
expanded_export = expanded_export.sort_values(
    ["approval_fy", "predicted_risk_score", "loan_row_id"],
    ascending=[True, False, True],
    kind="stable"
).reset_index(drop=True)

# Validate record counts and outcomes
cohort_check = (
    expanded_export
    .groupby("approval_fy")
    .agg(
        loan_count=("loan_row_id", "size"),
        chargeoffs=("chargeoff_within_5yr", "sum")
    )
)

display(cohort_check)

assert cohort_check.loc[2018, "loan_count"] == 54194
assert cohort_check.loc[2019, "loan_count"] == 45670
assert cohort_check.loc[2018, "chargeoffs"] == 2529
assert cohort_check.loc[2019, "chargeoffs"] == 1786

# Export the expanded dataset
export_path = Path(
    "../data/exports/sba_simulator_expanded.csv"
)

export_path.parent.mkdir(parents=True, exist_ok=True)
expanded_export.to_csv(export_path, index=False)

print("Expanded dataset saved:", export_path.exists())
print("Total records:", len(expanded_export))
print("Columns:", len(expanded_export.columns))

,loan_count,chargeoffs
approval_fy,,
2018,54194,2529
2019,45670,1786


Expanded dataset saved: True
Total records: 99864
Columns: 11


In [21]:
# Step 93.1 — Charge-off amount integrity audit

chargeoff_amount_audit = (
    expanded_export
    .groupby("approval_fy")
    .apply(
        lambda g: pd.Series({
            "total_loans": len(g),
            "five_year_events": int(
                g["chargeoff_within_5yr"].sum()
            ),
            "missing_amounts_for_events": int(
                g.loc[
                    g["chargeoff_within_5yr"] == 1,
                    "gross_chargeoff_amount"
                ].isna().sum()
            ),
            "zero_amounts_for_events": int(
                (
                    g.loc[
                        g["chargeoff_within_5yr"] == 1,
                        "gross_chargeoff_amount"
                    ] == 0
                ).sum()
            ),
            "five_year_event_dollars": (
                g.loc[
                    g["chargeoff_within_5yr"] == 1,
                    "gross_chargeoff_amount"
                ].sum()
            )
        })
    )
)

display(chargeoff_amount_audit.round(2))

/var/folders/g8/rnt7v8rs4474m76z7j66yz4h0000gn/T/ipykernel_82385/1962410902.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


,total_loans,five_year_events,missing_amounts_for_events,zero_amounts_for_events,five_year_event_dollars
approval_fy,,,,,
2018,54194.0,2529.0,0.0,0.0,2.895600e+08
2019,45670.0,1786.0,0.0,0.0,2.607024e+08
